In [ ]:
# SPDX-FileCopyrightText: 2024-2026 Dan J. Bower <dbower@eaps.ethz.ch>
# SPDX-FileCopyrightText: 2024-2026 Paolo A. Sossi
#
# SPDX-License-Identifier: Apache-2.0

import logging
from pathlib import Path
from typing import Literal

from atmodeller import (
    ChemicalSpecies,
    EquilibriumModel,
    PurePhase,
    ThermodynamicState,
    debug_logger,
)

logger = debug_logger()
logger.setLevel(logging.INFO)

# For more output use DEBUG
# logger.setLevel(logging.DEBUG)

# Gas Mixing

This notebook is available at `notebooks/examples/gas_mixing.ipynb` and is easiest to obtain by downloading the source code.

In gas mixing experiments, a predefined gaseous mixture is injected into a chamber and allowed to equilibrate thermodynamically under controlled temperature-pressure conditions. In this example, we consider a gas mixture that equilibrates in the presence of solid carbon (graphite).

In [ ]:
# Whether to export the data to Excel and pickle files, and where
WRITE_OUTPUT = False
OUTPUT_DIRECTORY = Path("output") / "gas_mixing"

# There are a few different output formats
output_format: Literal["elements_species", "named_arrays"] = "named_arrays"

if WRITE_OUTPUT:
    OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

# Define allowable gas species at equilibrium
H2_g = ChemicalSpecies.create_gas("H2")
N2_g = ChemicalSpecies.create_gas("N2")
CH4_g = ChemicalSpecies.create_gas("CH4")
CHN_g = ChemicalSpecies.create_gas("CHN")
H_g = ChemicalSpecies.create_gas("H")
gas_species = (H2_g, N2_g, CH4_g, CHN_g, H_g)

# Enforce graphite as a condensate (a pure phase in this model), so
# solve_for_stability=False
graphite = PurePhase.from_component("graphite", solve_for_stability=False)
condensate_phases = (graphite,)

# Set the thermodynamic state of the system
state = ThermodynamicState.from_species(
    gas_species, pressure=1, temperature=1773.15, melt_fraction=0, condensates=condensate_phases
)

# Define the mole fractions of input gases
mole_fractions = {"H2": 0.5, "N2": 0.5}

# Now define the model
model = EquilibriumModel.from_state(state, mass_constraints=mole_fractions, mass_units="moles")

# Solve with an automatic initial guess and default solver settings
output = model.solve_with_default()

# Optionally print solver stats to the logger
output.solver_stats_to_logger()

# Quick look at the solution
# output.quick_look()

# The abundance of C cannot be solved for, so the output amount (number of moles) of graphite
# should be ignored.

if WRITE_OUTPUT:
    # Write the complete solution to Excel
    output.to_excel(OUTPUT_DIRECTORY / "gas_mixing", output_format=output_format)

    # Write the data to a pickle file with dataframes
    output.to_pickle(OUTPUT_DIRECTORY / "gas_mixing", output_format=output_format)